# 03v2 — детектор без «выучивания» видео + честные таймлайны

**Что было не так в 03.** Кадры для обучения и проверки детектора брались из одних и тех же видео
(соседние кадры одного ролика попадали и в train, и в valid). Детектор фактически видел все 40 роликов,
поэтому mAP 0.79 завышен, а правила, проверенные на этих же видео (32/40), выглядят лучше, чем будут на тесте.

**Что делает этот ноутбук.**
1. Делит кадры **по видео** (дубликаты X-47 = X-4 и X-48 = X-5 всегда вместе) на 3 части (фолда).
2. Обучает детектор 3 раза: каждый раз на 2/3 роликов, проверяет на оставшейся 1/3, которую он **не видел**.
   Это честный mAP — то, чего ждать на новых видео.
3. Меньше переобучения: заморожен backbone (`freeze=10`), меньше эпох, сильные аугментации ракурса.
   Классы с 5–12 примерами (phone, cup, handbag) **не обучаем** — берём их из готовой COCO-модели.
4. Финальный детектор на всех данных → `models/cash_yolo11m_v2.pt` (для сервиса).
5. Таймлайны **out-of-fold**: каждое видео размечает детектор, который этого видео не видел →
   `timelines_oof/` — на них честно проверять правила. Плюс `timelines_v2/` от финального детектора.

Старые файлы (`cash_yolo11m_auto.pt`, `timelines/`) **не перезаписываются**.

**Время:** ~45–60 мин (3 фолда + финал по ~10–15 мин, таймлайны ~6 мин).
**Перед запуском:** освободите GPU — Kernel → Shut Down All Kernels в других ноутбуках, сервис на 8000 остановить.
Запуск: **Run → Run All Cells**.

## 0. Параметры

In [ ]:
import os
from pathlib import Path

WORK   = Path(os.getenv("WORK", "/workspace"))
DATA   = WORK / "data"
MODE   = os.getenv("MODE", "auto")              # "auto" (авторазметка v2) | "cvat" (ручная) | "draft"
DEVICE = os.getenv("DEVICE", "0")

DATASET_ZIP = DATA / "datathon26_detector_dataset.zip"
VIDEOS_DIR  = DATA / "open" / "videos"
LABELS_XLSX = DATA / "open" / "open_labels.xlsx"


def _first(*paths):
    return next((p for p in paths if p.exists()), paths[0])


COCO_SRC = {
    "cvat":  {"train": _first(DATA / "labels" / "train_ls.json", DATA / "cvat" / "train_coco.zip"),
              "valid": _first(DATA / "labels" / "valid_ls.json", DATA / "cvat" / "valid_coco.zip")},
    "draft": {"train": DATA / "prelabels" / "train_p1_coco1.0.zip", "valid": DATA / "prelabels" / "valid_coco1.0.zip"},
    "auto":  {"train": DATA / "autolabels" / "train_auto.zip", "valid": DATA / "autolabels" / "valid_auto.zip"},
}[MODE]

# классы в файлах разметки (как в 03)
ALL_CLASSES = ["cash", "collection_bag", "safe_open", "safe_closed", "counting_machine",
               "id_document", "phone", "handbag", "cup"]
# обучаем только классы, где достаточно примеров; телефон/чашка/сумка/ноутбук — из COCO-модели yolo11m
TRAIN_CLASSES = ["cash", "collection_bag", "safe_open", "safe_closed", "counting_machine"]
COCO_EXTRA = {67: "phone", 41: "cup", 26: "handbag", 24: "handbag", 63: "laptop"}

BASE_MODEL = os.getenv("BASE_MODEL", "yolo11m.pt")
N_FOLDS  = int(os.getenv("N_FOLDS", 3))
EPOCHS   = int(os.getenv("EPOCHS", 40))
PATIENCE = 10
FREEZE   = 10          # заморозить backbone: на ~800 кадрах учим только «голову» -> меньше запоминания фона
IMGSZ    = 896
BATCH    = 16
SEED     = 42
TIMELINE_FPS = 4.0
DUP = {"X-47": "X-4", "X-48": "X-5"}   # побайтно одинаковые видео — всегда в одном фолде

RUNS, MODELS = WORK / "runs", WORK / "models"
DS2 = WORK / "yolo_ds_v2"
TL_OOF, TL_FINAL = WORK / "timelines_oof", WORK / "timelines_v2"
for p in (RUNS, MODELS, DS2, TL_OOF, TL_FINAL):
    p.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

## 1. Окружение

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.180", "pandas", "matplotlib",
                    "supervision", "av", "openpyxl", "pyarrow", "lap"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install не прошёл, см. вывод выше"
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free {free/1e9:.1f} / {total/1e9:.1f} GB")
    assert free > 8e9, "GPU занята (меньше 8 ГБ свободно): остановите сервис и другие ноутбуки с Qwen"

In [ ]:
import json, re, shutil, time, unicodedata, zipfile
from collections import defaultdict
from urllib.parse import unquote

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


def nfc(s):
    return unicodedata.normalize("NFC", str(s))


def norm_vid(name):
    """'Х-12' (кириллица) / 'X-12' / 'Х-12.mov' -> 'X-12'."""
    s = nfc(name).strip().replace("Х", "X").replace("х", "X").replace("x", "X")
    return re.sub(r"\.mov$", "", s, flags=re.I)


def video_of_frame(name):
    """'X-34_t0015.12.jpg' -> 'X-34'; дубликаты сливаются (X-47 -> X-4)."""
    m = re.match(r"\s*[XХxх]\s*[-_]?\s*(\d+)", nfc(name))
    if not m:
        return None
    v = f"X-{int(m.group(1))}"
    return DUP.get(v, v)


def unzip_utf8(src, dst):
    """Распаковка с кириллицей в именах."""
    with zipfile.ZipFile(src) as z:
        for info in z.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:
                try:
                    name = name.encode("cp437").decode("utf-8")
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass
            target = Path(dst) / nfc(name)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as fsrc, open(target, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst)


def load_labels():
    df = pd.read_excel(LABELS_XLSX)
    df.columns = ["file", "ts", "label"]

    def parse_ts(v):
        if pd.isna(v):
            return None
        s = str(v).lower(); mm = re.search(r"(\d+)\s*мин", s); ss = re.search(r"(\d+)\s*сек", s)
        return float((int(mm.group(1)) * 60 if mm else 0) + (int(ss.group(1)) if ss else 0))
    df["video_id"] = df.file.map(norm_vid)
    df["violation"] = df.label.str.strip().str.lower().eq("нарушение")
    df["t_sec"] = df.ts.map(parse_ts)
    files = {norm_vid(p.name): p for p in VIDEOS_DIR.glob("*.mov")}
    df["path"] = df.video_id.map(files)
    return df[["video_id", "violation", "t_sec", "path"]]

## 2. Данные и проверка утечки в старом разбиении

In [ ]:
DS = DATA / "detds"
if not (DS / "dataset" / "manifest.csv").exists():
    unzip_utf8(DATASET_ZIP, DS)
if not VIDEOS_DIR.exists() and (DATA / "open.zip").exists():
    unzip_utf8(DATA / "open.zip", DATA)
missing = [p for p in COCO_SRC.values() if not Path(p).exists()]
assert not missing, f"нет файлов разметки: {missing}"

CLASS_ID = {c: i + 1 for i, c in enumerate(ALL_CLASSES)}


def ls_to_coco(tasks):
    """Label Studio JSON -> COCO (только задачи с сохранённой разметкой)."""
    coco = {"images": [], "annotations": [], "categories": [{"id": i, "name": c} for c, i in CLASS_ID.items()]}
    for task in tasks:
        done = [a for a in task.get("annotations", []) if not a.get("was_cancelled")]
        if not done:
            continue
        ann = max(done, key=lambda a: a.get("updated_at") or a.get("created_at") or "")
        d = task["data"]; img_id = len(coco["images"]) + 1
        coco["images"].append({"id": img_id, "file_name": d["fname"], "width": d["width"], "height": d["height"],
                               "reviewed_empty": not ann["result"]})
        for r in ann["result"]:
            if r.get("type") != "rectanglelabels":
                continue
            v, W, H = r["value"], r.get("original_width", d["width"]), r.get("original_height", d["height"])
            for lbl in v["rectanglelabels"]:
                if lbl in CLASS_ID:
                    coco["annotations"].append({"id": len(coco["annotations"]) + 1, "image_id": img_id,
                                                "category_id": CLASS_ID[lbl],
                                                "bbox": [v["x"] * W / 100, v["y"] * H / 100,
                                                         v["width"] * W / 100, v["height"] * H / 100]})
    return coco


def load_coco(src):
    src = Path(src)
    if src.suffix == ".zip":
        with zipfile.ZipFile(src) as z:
            name = next(n for n in z.namelist() if n.endswith(".json"))
            data = json.loads(z.read(name))
    else:
        data = json.loads(src.read_text())
    return ls_to_coco(data) if isinstance(data, list) else data


SRC_INDEX = {nfc(p.name): p for p in (DS / "dataset" / "images").rglob("*.jpg")}
frames = {}       # имя кадра -> {src, video, split0, lines}
for split0, src in COCO_SRC.items():
    coco = load_coco(src)
    cat = {c["id"]: c["name"] for c in coco["categories"]}
    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)
    for im in coco["images"]:
        name = nfc(Path(unquote(im["file_name"])).name)
        if name not in SRC_INDEX:
            continue
        W, H = im["width"], im["height"]
        lines, n_any = [], 0
        for a in anns[im["id"]]:
            cname = cat.get(a["category_id"])
            x, y, w, h = a["bbox"]
            if w <= 1 or h <= 1:
                continue
            n_any += 1
            if cname not in TRAIN_CLASSES:
                continue
            x0, y0 = max(0, x), max(0, y); x1, y1 = min(W, x + w), min(H, y + h)
            lines.append(f"{TRAIN_CLASSES.index(cname)} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
        if n_any == 0 and not im.get("reviewed_empty"):
            continue          # кадр без разметки вообще = не размечен, не берём
        frames[name] = {"src": SRC_INDEX[name], "video": video_of_frame(name), "split0": split0, "lines": lines}

fr = pd.DataFrame([{"name": k, "video": v["video"], "split0": v["split0"], "n_boxes": len(v["lines"])}
                   for k, v in frames.items()])
bad = fr[fr.video.isna()]
assert bad.empty, f"не удалось определить видео по имени кадра: {bad.name.head().tolist()}"
print("кадров:", len(fr), "| видео:", fr.video.nunique(), "| пример имени:", fr.name.iloc[0])
leak = fr.groupby("video").split0.nunique()
if (leak > 1).any():
    print(f"⚠️ УТЕЧКА в старом разбиении: у {int((leak > 1).sum())} из {len(leak)} видео кадры есть и в train, "
          "и в valid -> mAP 0.79 из ноутбука 03 завышен.")
else:
    print("старое разбиение было по видео — утечки по кадрам нет")

## 3. Фолды по видео

In [ ]:
lab = load_labels()
viol = {DUP.get(v, v): bool(x) for v, x in zip(lab.video_id, lab.violation)}
vids = sorted(fr.video.unique(), key=lambda v: int(v.split("-")[1]))
rng = np.random.default_rng(SEED)
pos = [v for v in vids if viol.get(v)]
neg = [v for v in vids if not viol.get(v)]
rng.shuffle(pos); rng.shuffle(neg)
FOLD_OF = {v: i % N_FOLDS for i, v in enumerate(pos + neg)}          # поровну видео с нарушением и без
FOLD_OF.update({d: FOLD_OF[o] for d, o in DUP.items() if o in FOLD_OF})
fr["fold"] = fr.video.map(FOLD_OF)
display(fr.groupby("fold").agg(videos=("video", "nunique"), frames=("name", "size"), boxes=("n_boxes", "sum")))
(MODELS / "folds_v2.json").write_text(json.dumps(FOLD_OF, ensure_ascii=False, indent=1))

# одна копия картинок/разметки; фолды — просто списки путей
img_dir, lbl_dir = DS2 / "images" / "all", DS2 / "labels" / "all"
for d in (img_dir, lbl_dir):
    d.mkdir(parents=True, exist_ok=True)
for name, f in frames.items():
    if not (img_dir / name).exists():
        shutil.copy2(f["src"], img_dir / name)
    (lbl_dir / (Path(name).stem + ".txt")).write_text("\n".join(f["lines"]))
NAMES_YAML = "".join(f"  {i}: {c}\n" for i, c in enumerate(TRAIN_CLASSES))


def write_split(tag, train_names, val_names):
    (DS2 / f"{tag}_train.txt").write_text("\n".join(str(img_dir / n) for n in train_names))
    (DS2 / f"{tag}_val.txt").write_text("\n".join(str(img_dir / n) for n in val_names))
    y = DS2 / f"{tag}.yaml"
    y.write_text(f"path: {DS2}\ntrain: {tag}_train.txt\nval: {tag}_val.txt\nnames:\n{NAMES_YAML}")
    return y


YAML = {f"fold{k}": write_split(f"fold{k}", fr[fr.fold != k].name, fr[fr.fold == k].name) for k in range(N_FOLDS)}
YAML["final"] = write_split("final", fr.name, fr[fr.fold == 0].name)   # val финала только для логов
for k in range(N_FOLDS):
    print(f"фолд {k}: проверочные видео ->", sorted(fr[fr.fold == k].video.unique(), key=lambda v: int(v[2:])))

## 4. Обучение по фолдам → честный mAP на невиденных видео

In [ ]:
from ultralytics import YOLO

AUG = dict(degrees=15, translate=0.15, scale=0.5, shear=5, perspective=0.0007, fliplr=0.5,   # другой ракурс
           hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,                                                 # другое освещение
           mosaic=1.0, mixup=0.1, close_mosaic=5)


def train(tag):
    t0 = time.time()
    YOLO(BASE_MODEL).train(data=str(YAML[tag]), epochs=EPOCHS, patience=PATIENCE, freeze=FREEZE, imgsz=IMGSZ,
                           batch=BATCH, device=DEVICE, seed=SEED, project=str(RUNS), name=f"det_v2_{tag}",
                           exist_ok=True, cos_lr=True, cache="ram", workers=8, plots=False, verbose=False, **AUG)
    print(f"[{tag}] обучение заняло {(time.time() - t0) / 60:.1f} мин", flush=True)
    return RUNS / f"det_v2_{tag}" / "weights" / "best.pt"


def map50_by_class(weights, yaml, name):
    m = YOLO(str(weights)).val(data=str(yaml), imgsz=IMGSZ, device=DEVICE, split="val", plots=False,
                               verbose=False, project=str(RUNS), name=name, exist_ok=True)
    row = {TRAIN_CLASSES[int(ci)]: m.box.class_result(k)[2] for k, ci in enumerate(m.box.ap_class_index)}
    row["ВСЕ"] = m.box.map50
    return row


FOLD_W, oof = {}, {}
for k in range(N_FOLDS):
    FOLD_W[k] = MODELS / f"cash_v2_fold{k}.pt"
    shutil.copy2(train(f"fold{k}"), FOLD_W[k])
    oof[f"фолд {k}"] = map50_by_class(FOLD_W[k], YAML[f"fold{k}"], f"det_v2_fold{k}_val")
oof_tbl = pd.DataFrame(oof).T
oof_tbl.loc["СРЕДНЕЕ"] = oof_tbl.mean()
display(oof_tbl.round(3))
print(f"Честный mAP50 на НЕВИДЕННЫХ видео: {oof_tbl.loc['СРЕДНЕЕ', 'ВСЕ']:.3f}  "
      f"(в 03 было 0.793 — на кадрах из уже знакомых видео)")
oof_tbl.to_csv(RUNS / "det_v2_oof_map50.csv")

### Прокси-тест на другой ракурс (на невиденных видео)

In [ ]:
def shifted_yaml(tag, img_paths, angle_max=12, seed=0):
    rng_ = np.random.default_rng(seed)
    root = WORK / "yolo_ds_v2_shift" / tag
    for sub in ("images/val", "labels/val"):
        shutil.rmtree(root / sub, ignore_errors=True); (root / sub).mkdir(parents=True)
    for p in map(Path, img_paths):
        img = cv2.imread(str(p)); H, W = img.shape[:2]
        a, s = rng_.uniform(-angle_max, angle_max), rng_.uniform(0.8, 1.1)
        M = cv2.getRotationMatrix2D((W / 2, H / 2), a, s)
        M[:, 2] += rng_.uniform(-0.05, 0.05, 2) * (W, H)
        out = cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)
        lines = []
        for line in (lbl_dir / (p.stem + ".txt")).read_text().splitlines():
            c, cx, cy, w, h = line.split(); cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
            pts = np.array([[cx-w/2, cy-h/2, 1], [cx+w/2, cy-h/2, 1], [cx+w/2, cy+h/2, 1], [cx-w/2, cy+h/2, 1]]) @ M.T
            x0, y0 = pts.min(0).clip(0, [W, H]); x1, y1 = pts.max(0).clip(0, [W, H])
            if (x1 - x0) > 3 and (y1 - y0) > 3:
                lines.append(f"{c} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
        cv2.imwrite(str(root / "images/val" / p.name), out)
        (root / "labels/val" / (p.stem + ".txt")).write_text("\n".join(lines))
    y = root / "data.yaml"
    y.write_text(f"path: {root}\ntrain: images/val\nval: images/val\nnames:\n{NAMES_YAML}")
    return y


shift = {}
for k in range(N_FOLDS):
    vals = [s for s in (DS2 / f"fold{k}_val.txt").read_text().split("\n") if s]
    shift[f"фолд {k}"] = map50_by_class(FOLD_W[k], shifted_yaml(f"fold{k}", vals, seed=k), f"det_v2_fold{k}_shift")["ВСЕ"]
cmp = pd.DataFrame({"невиденные видео": oof_tbl["ВСЕ"].drop("СРЕДНЕЕ"), "+ поворот/масштаб": pd.Series(shift)})
cmp.loc["СРЕДНЕЕ"] = cmp.mean()
display(cmp.round(3))

## 5. Финальный детектор на всех видео (для сервиса)

In [ ]:
FINAL_W = MODELS / "cash_yolo11m_v2.pt"
shutil.copy2(train("final"), FINAL_W)
print("финальные веса:", FINAL_W, "| старые веса cash_yolo11m_auto.pt не тронуты")

## 6. Таймлайны

- `timelines_oof/` — каждое видео размечено детектором **своего фолда** (этого видео он не видел). На них честно
  проверять правила (`04_rules_eval`).
- `timelines_v2/` — финальный детектор (как будет в сервисе).
- Телефон / чашка / сумка / ноутбук теперь из COCO-модели (обучена на миллионах картинок, не на наших видео),
  колонка `src` = `coco` или `custom`. Формат parquet прежний.

In [ ]:
labels = load_labels()
assert labels.path.notna().all(), labels[labels.path.isna()]
person_model = YOLO("yolo11m.pt")
pose_model = YOLO("yolo11m-pose.pt")
fold_models = {k: YOLO(str(FOLD_W[k])) for k in range(N_FOLDS)}
final_model = YOLO(str(FINAL_W))
KP = {"nose": 0, "l_wrist": 9, "r_wrist": 10, "l_hip": 11, "r_hip": 12}


def video_timeline(path, obj_model):
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; cap.release()
    step = max(1, round(fps / TIMELINE_FPS))
    common = dict(stream=True, vid_stride=step, device=DEVICE, verbose=False, imgsz=IMGSZ)
    persons, pose, objects = [], [], []
    for k, r in enumerate(person_model.track(source=str(path), classes=[0, *COCO_EXTRA], conf=0.3,
                                             tracker="botsort.yaml", **common)):
        t = k * step / fps
        ids = r.boxes.id.int().cpu().tolist() if r.boxes.id is not None else [-1] * len(r.boxes)
        for tid, b, c, cl in zip(ids, r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            if int(cl) == 0:
                persons.append({"t": t, "track": tid, "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3], "conf": float(c)})
            else:
                objects.append({"t": t, "cls": COCO_EXTRA[int(cl)], "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3],
                                "conf": float(c), "src": "coco"})
    for k, r in enumerate(pose_model.predict(source=str(path), conf=0.3, **common)):
        t = k * step / fps
        if r.keypoints is None:
            continue
        for b, kp in zip(r.boxes.xyxy.cpu().numpy(), r.keypoints.data.cpu().numpy()):
            row = {"t": t, "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3]}
            for name, i in KP.items():
                row[f"{name}_x"], row[f"{name}_y"], row[f"{name}_c"] = kp[i]
            pose.append(row)
    for k, r in enumerate(obj_model.predict(source=str(path), conf=0.25, **common)):
        t = k * step / fps
        for b, c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            objects.append({"t": t, "cls": obj_model.names[int(c)], "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3],
                            "conf": float(s), "src": "custom"})
    return pd.DataFrame(persons), pd.DataFrame(pose), pd.DataFrame(objects)


timing = []
for r in labels.itertuples():
    k = FOLD_OF.get(r.video_id, 0)
    for tag, model_, out in (("oof", fold_models[k], TL_OOF), ("final", final_model, TL_FINAL)):
        t0 = time.time()
        P, K, O = video_timeline(r.path, model_)
        for name, df_ in (("persons", P), ("pose", K), ("objects", O)):
            df_.to_parquet(out / f"{r.video_id}_{name}.parquet", index=False)
        timing.append({"video_id": r.video_id, "set": tag, "fold": k, "sec": round(time.time() - t0, 1),
                       "persons": len(P), "objects": len(O)})
    print(r.video_id, "фолд", k, "|", timing[-2]["sec"], "c /", timing[-1]["sec"], "c", flush=True)
timing = pd.DataFrame(timing)
timing.to_csv(TL_OOF / "timing.csv", index=False)
print(f"время на видео (финальный детектор): p50 {timing[timing.set == 'final'].sec.median():.1f} c, "
      f"макс {timing[timing.set == 'final'].sec.max():.1f} c")

## 7. Итог и что дальше

In [ ]:
print("Честный mAP50 детектора на невиденных видео:", round(float(oof_tbl.loc["СРЕДНЕЕ", "ВСЕ"]), 3))
print("  с поворотом/масштабом:                    ", round(float(cmp.loc["СРЕДНЕЕ", "+ поворот/масштаб"]), 3))
print()
print("1) Честная проверка правил: в 04_rules_eval.ipynb поставьте  TIMELINES = WORK / 'timelines_oof'")
print("   Итог 04 на них = чего реально ждать на новых видео. Сравните со старыми таймлайнами.")
print("2) Сервис: object_weights = WORK / 'models' / 'cash_yolo11m_v2.pt'")
print("   (в сервисе и в 11_tta Analyzer(...): object_weights=str(WORK / 'models' / 'cash_yolo11m_v2.pt'))")
print("3) Разбиение по фолдам сохранено в", MODELS / "folds_v2.json")